# TrafficVision-AI :: Notebook 03 — MLOps Pipeline

---
**Version:** 2.0.0 | **Scope:** End-to-End MLOps Lifecycle, CI/CD, Model Registry, Monitoring

---

## Table of Contents
1. [MLOps Philosophy](#1-mlops-philosophy)
2. [Experiment Tracking with MLflow](#2-experiment-tracking)
3. [Model Registry & Promotion](#3-model-registry)
4. [CI/CD Pipeline Architecture](#4-cicd-pipeline)
5. [Drift Detection in Production](#5-drift-detection)
6. [Automated Retraining Triggers](#6-automated-retraining)
7. [Data Version Control (DVC)](#7-dvc)
8. [Airflow DAG Design](#8-airflow-dag)
9. [Monitoring Stack](#9-monitoring-stack)
10. [Production Readiness Checklist](#10-checklist)

## 1. MLOps Philosophy

TrafficVision-AI implements **MLOps Level 2** — fully automated ML pipeline with:

```
MLOps Maturity Model
=====================

Level 0  Manual process
         └── Notebooks only, no deployment

Level 1  ML pipeline automation
         └── Automated training, manual deployment

Level 2  CI/CD pipeline automation  ← TrafficVision-AI
         ├── Automated training pipeline
         ├── Automated model evaluation
         ├── Automated deployment on quality gate pass
         ├── Continuous monitoring & drift detection
         └── Automated retraining triggers

Level 3  Full self-healing ML system
         └── Online learning, zero-touch ops
```

### Core MLOps Principles Applied

| Principle | Implementation |
|-----------|----------------|
| Reproducibility | DVC data versioning + MLflow param logging |
| Automation | GitHub Actions 7-stage CI/CD |
| Versioning | Semantic versioning for models + code |
| Testing | 85%+ code coverage + integration smoke tests |
| Monitoring | PSI drift detection + Prometheus metrics |
| Rollback | ModelRegistry.promote() with archived stage |

## 2. Experiment Tracking with MLflow

```
MLflow Architecture
===================

  Training Job
       │
       │ mlflow.log_params(backbone, lr, batch_size)
       │ mlflow.log_metrics(val_loss, mean_iou per epoch)
       │ mlflow.keras.log_model(model)
       ▼
  MLflow Tracking Server
  ├── PostgreSQL (run metadata, params, metrics)
  └── S3 / local (model artifacts, checkpoints)
       │
       ▼
  MLflow Model Registry
  ├── Staging   ← candidate models
  └── Production ← serving models
```

### Experiment Comparison Strategy

For each backbone × loss function × augmentation configuration:
- Run ID logged with full hyperparameter set
- Training curves saved as artifacts
- Grad-CAM samples stored per run
- Model artifact registered if IoU > 0.60

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Demonstrate MLflow logging (without running actual training)
print('MLflow experiment tracking setup:')
print('  Tracking URI : http://localhost:5000')
print('  Experiment   : trafficvision-detection')
print('  Params logged: backbone, lr, batch_size, dense_units, dropout_rate')
print('  Metrics logged: train_loss, val_loss, val_mae, mean_iou (per epoch)')
print('  Artifacts    : model.keras, eval_report.json, gradcam_samples/')

mlflow_example = '''
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("trafficvision-detection")

with mlflow.start_run(run_name="resnet50_phase2_v2"):
    mlflow.log_params({
        "backbone": "resnet50",
        "learning_rate": 1e-4,
        "batch_size": 32,
        "epochs_phase1": 30,
        "epochs_phase2": 20,
        "loss": "combined",
    })
    
    # ... training ...
    
    mlflow.log_metrics({
        "best_val_loss": 0.0312,
        "test_mae": 0.035,
        "mean_iou": 0.68,
    })
    mlflow.keras.log_model(keras_model, "model")
    mlflow.log_artifact("models/evaluation/eval_report.json")
'''
print(mlflow_example)

## 3. Model Registry & Promotion

```mermaid
graph LR
    A[Training Run<br/>IoU=0.73] -->|register| B[Experimental]
    B -->|IoU>0.70<br/>auto-promote| C[Staging]
    C -->|Manual approval<br/>+ smoke tests| D[Production]
    D -->|new version<br/>deployed| E[Archived]
    
    style D fill:#22c55e,color:#fff
    style C fill:#eab308,color:#000
    style B fill:#6b7280,color:#fff
    style E fill:#374151,color:#fff
```

### Promotion Gates

| Stage | Gate | Automated? |
|-------|------|------------|
| Experimental → Staging | IoU > 0.70 | ✅ Yes |
| Staging → Production | Human approval + IoU > 0.73 + latency < 200ms | ⚠️ Partially |
| Production → Archived | New production version exists | ✅ Yes |

In [ ]:
# Demonstrate Model Registry usage
from src.services.model_registry import ModelRegistry, ModelStage
from pathlib import Path
import tempfile, json

registry_dir = Path(tempfile.mkdtemp())
registry = ModelRegistry(registry_dir)
print('Model Registry initialized at:', registry_dir)
print('Registry state:', registry)

## 4. CI/CD Pipeline Architecture

```
CI/CD WORKFLOW (GitHub Actions)
================================

  git push / PR
       │
       ▼
  ┌─────────────────────────────────┐
  │  Stage 1: Lint (2 min)          │
  │  ruff + black + mypy             │
  └─────────────┬───────────────────┘
                │ PASS
                ▼
  ┌─────────────────────────────────┐
  │  Stage 2: Test (5 min)          │
  │  pytest + coverage > 80%        │
  └─────────────┬───────────────────┘
                │ PASS
                ▼
  ┌─────────────────────────────────┐
  │  Stage 3: Security (3 min)      │
  │  bandit + safety + trivy        │
  └─────────────┬───────────────────┘
                │ PASS
                ▼
  ┌─────────────────────────────────┐
  │  Stage 4: Docker Build (8 min)  │
  │  multi-arch amd64 + arm64       │
  └─────────────┬───────────────────┘
                │ PASS
                ▼
  ┌─────────────────────────────────┐
  │  Stage 5: Integration (4 min)   │
  │  smoke tests on live container  │
  └─────────────┬───────────────────┘
                │ PASS
          ┌─────┴───────┐
          │             │
   develop branch    version tag
          │             │
          ▼             ▼
    Staging Deploy  Production Deploy
    (automatic)     (manual gate)
```

**Total pipeline time: ~22 minutes for a full production deployment**

Key optimizations:
- Layer caching in Docker build (8→3 min on cache hit)
- Parallel test matrix (py3.10 + py3.11 in parallel)
- Incremental coverage upload (only changed files)

## 5. Drift Detection in Production

```
DRIFT MONITORING ARCHITECTURE
==============================

  Production Inference
       │
       │ every prediction
       ▼
  ModelMonitor.record(bbox_pred, latency_ms)
  [sliding window deque, N=1000]
       │
       │ every 100 requests
       ▼
  Drift Computation
  ├── PSI (Population Stability Index) per coordinate
  │     < 0.10 → ✅ stable
  │     0.10–0.20 → ⚠️ alert
  │     > 0.20 → 🔴 retrain trigger
  │
  ├── KL Divergence
  │
  └── Latency Percentiles (p50, p95, p99)
       │
       ▼
  DriftReport → Prometheus metric → Grafana alert
  DriftReport → JSON file → S3 audit trail
  PSI > 0.20 → Airflow DAG trigger (retraining pipeline)
```

**Reference Distribution**: Saved from test-set predictions at training time.
Updated on each successful production promotion.

## 9. Monitoring Stack

```
OBSERVABILITY ARCHITECTURE
============================

  FastAPI Application
  ├── /metrics          ← Prometheus scrape endpoint
  ├── Structured JSON logs → stdout → Fluent Bit → Elasticsearch
  └── Distributed traces → Jaeger (optional)
         │
         ▼
  Prometheus (15s scrape)
  ├── trafficvision_requests_total (counter)
  ├── trafficvision_request_latency_seconds (histogram)
  ├── trafficvision_cache_hit_ratio (gauge)
  ├── trafficvision_model_drift_psi (gauge, per coord)
  └── trafficvision_active_connections (gauge)
         │
         ▼
  Grafana (dashboards)
  ├── Service Health Dashboard
  ├── ML Performance Dashboard (IoU over time)
  ├── Drift Monitoring Dashboard
  └── Infrastructure Dashboard
```

## 10. Production Readiness Checklist

| Category | Item | Status |
|----------|------|--------|
| **Testing** | Unit test coverage > 80% | ✅ |
| **Testing** | Integration tests in CI | ✅ |
| **Security** | Non-root container user | ✅ |
| **Security** | TLS on all endpoints | ✅ |
| **Security** | Secrets via manager (not env vars) | ✅ |
| **Observability** | Structured JSON logging | ✅ |
| **Observability** | Prometheus metrics | ✅ |
| **Observability** | Health + readiness probes | ✅ |
| **Reliability** | HPA (3–20 replicas) | ✅ |
| **Reliability** | PodDisruptionBudget | ✅ |
| **MLOps** | Model versioning | ✅ |
| **MLOps** | Drift detection | ✅ |
| **MLOps** | Automated retraining trigger | ✅ |
| **Documentation** | OpenAPI / Swagger | ✅ |

---
*TrafficVision-AI MLOps Notebook — Enterprise R&D Documentation*